<a href="https://colab.research.google.com/github/Ymin-2/ESAA/blob/main/ESAA_OB_WEEK03_1_review3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**주제**

제2회 코스포 x 데이콘 도서 추천 알고리즘 AI경진대회 채용_도서 추천 알고리즘 AI 모델 개발  
https://dacon.io/competitions/official/236093/codeshare/8405?page=1&dtype=recent

##**데이터**

- train.csv
  - ID : 샘플 고유 ID
  - User-ID : 유저 고유 ID
  - Book-ID : 도서 고유 ID
  - 유저 정보
    - Age : 나이
    - Location : 지역
  - 도서 정보
    - Book-Title : 도서 명
    - Book-Author : 도서 저자
    - Year-Of-Publication : 도서 출판 년도 (-1일 경우 결측 혹은 알 수 없음)
    - Publisher : 출판사
  - Book-Rating : 유저가 도서에 부여한 평점 (0점 ~ 10점)
    - 단, 0점인 경우에는 유저가 해당 도서에 관심이 없고 관련이 없는 경우

- test.csv
  - ID : 샘플 고유 ID
  - User-ID : 유저 고유 ID
  - Book-ID : 도서 고유 ID
  - 유저 정보
    - Age : 나이
    - Location : 지역
  - 도서 정보
    - Book-Title : 도서 명
    - Book-Author : 도서 저자
    - Year-Of-Publication : 도서 출판 년도 (-1일 경우 결측 혹은 알 수 없음)
    - Publisher : 출판사

- sample_submission.csv
  - ID : 샘플 고유 ID
  - Book-Rating : 예측한 유저가 도서에 부여할 평점
    - 단, 0점인 경우에는 유저가 해당 도서에 관심이 없고 관련이 없는 경우

##**코드리뷰**

[전처리]
- Age, Year-Of-Publication: 10단위로 묶음, -1 등 이상치 제거하지 않음
- Location: ','로 나누어 가장 마지막 문자열을 country로 사용, 오탈자로 판단되는 것들 전처리
- Book-Title, Book-Author, Publisher: 특수문자, 공백 등 전처리, 그룹화 될 수 있도록 텍스트 전처리
- User-ID, Book-ID: 제거하지 않고 사용

[모델비교]
- 전처리+GridSearch를 통해 learning rate을 0.1로 고정, K-Fold 10 적용했을 때 가장 좋은 성능을 보임
- CatBoostRegressor와 LGBMRegressor로 예측한 결과를 평균내었을 때, 성능이 저하됨
  - LGBM 모델이 score를 낮게 예측하는 경향이 있기 때문으로 판단, 모델 제

##**배울점**

- 문자열 전처리가 성능에 중요하다
- Location 원본을 쓰지 않고 country로 단순화했다
- 결측/애매한 값을 그냥 두지 않고 규칙적으로 보정했다

In [ ]:
# 문자열 전처리
def text_preprocessing(summary):
    summary = re.sub("[.,\'\"''""!?' ']", "", summary)
    summary = re.sub("[^0-9a-zA-Z\\s]", "", summary)
    summary = summary.lower()
    return summary

# 저자명 통합
def process_book_author(book_author:str):
    # 셰익스피어
    # jkrowling
    if book_author in ['shakespeare','shakespere']:
        return 'williamshakespeare'
    elif book_author in ['joannekrowling','rowlingjk']:
        return 'jkrowling'
    elif 'disney' in book_author :
        return 'disney'

    return book_author

In [ ]:
# country 정리
def process_country(country:str):
    # usa, usofa, usacurrentlylivinginengland,
    if country in ['us','usofa','usacurrentlylivinginengland','unitedstate','unitedstates','hungaryandusa','unitedsates','america']:
        return 'usa'
    elif country in ['england','unitedkingdom','unitedkindgonm']:
        return 'uk'
    elif country in ['unitedarabemirates']:
        return 'uae'
    elif country in ["" , "na"] :
        return 'other'
    elif 'spain' in country :
        return 'spain'
    elif 'guinea' in country:
        return 'guinea'
    return country

# Book-ID 기반 country 보정
book_country_info = train_df.groupby(['Book-ID', 'preprocess_country'])['preprocess_country'].count().groupby('Book-ID').idxmax().reset_index()
book_country_info['preprocess_country'] = book_country_info['preprocess_country'].apply(lambda x : x[1])
replacement_values = book_country_info[['Book-ID', 'preprocess_country']].set_index('Book-ID')['preprocess_country']
train_df.loc[train_mask, 'preprocess_country'] = train_df.loc[train_mask, 'Book-ID'].map(replacement_values)

# 영어권 국가 feature 생성
train_df['is_en'] = 0
train_df.loc[train_df['preprocess_country'].isin(['usa','canada','uk']),'is_en'] = 1

test_df['is_en'] = 0
test_df.loc[test_df['preprocess_country'].isin(['usa','canada','uk']),'is_en'] = 1

In [ ]:
# CatBoost 파라미터
cb_learn_rate = 0.1
# cb_learn_rate = 0.006
# n_iterations = 40000
# n_iterations = 10000
n_iterations = 10000

early_stop_rounds = 400

opt_catboost_params = {'iterations' : n_iterations,
                       'learning_rate' : cb_learn_rate,
                       'bootstrap_type' : 'Bernoulli',
                       'loss_function' : 'RMSE',
                       'eval_metric' : 'RMSE',
                       'task_type' : 'GPU',
                       'od_type' : 'IncToDec',
                       'od_wait' : 100,
                       'verbose' : 500,
                       'random_seed' : 42}

# CatBoost 학습
clf = CatBoostRegressor(**opt_catboost_params)
# clf = CatBoostRegressor(**grid.best_params_)

clf.fit(X_train, y_train,
        cat_features=cat_col_list,
        eval_set=(X_val, y_val),
        use_best_model=True, plot=True,
)

print('CatBoost model is fitted: ' + str(clf.is_fitted()))
print('CatBoost model parameters:')
print(clf.get_params())

In [ ]:
# 10-fold KFold 앙상블
model_path = './model3'
os.makedirs(model_path, exist_ok=True)

kfold = KFold(n_splits = 10, random_state =42 , shuffle = True )

cb_pred = np.zeros((test_df.shape[0]))
cb_mae = 0
print(opt_catboost_params)
for i, idx in enumerate(kfold.split(X,y)):

    train_X, train_y = X.loc[idx[0]], y.loc[idx[0]]
    valid_X, valid_y = X.loc[idx[1]], y.loc[idx[1]]

    model = CatBoostRegressor(**opt_catboost_params)
#     model = CatBoostRegressor(**grid.best_params_)
    model.fit(train_X,train_y, cat_features=cat_col_list,
              eval_set = (valid_X,valid_y), early_stopping_rounds = 1000, verbose = 500,plot=True)
              # eval_set = [(train_X,train_y),(valid_X,valid_y)], early_stopping_rounds = 1000, verbose = 500)

    valid_pred = model.predict(valid_X)
    valid_mae = mean_absolute_error(valid_y, valid_pred)
    cb_mae += valid_mae /kfold.n_splits
    print(f"{i + 1} Fold MAE : {valid_mae}")

    fold_pred = model.predict(test_df[target_column]) / kfold.n_splits
    cb_pred += fold_pred

    model_path = './model3/new_catboostreg_{}'.format(i)
    model.save_model(model_path)

print(f"\n{model.__class__.__name__} AVG of MAE : {cb_mae}")